# story — storyboard / presentation mode

Builds a click-through presentation with multiple slides. Each slide controls:
- which layers are visible
- what the viewport shows (`"fit"` or an explicit `(pan_x, pan_y, zoom)` tuple)
- an optional title displayed in the window title bar

```python
story(
    data=None,      # Nx10 float32 instance array (single layer)
    layers=None,    # list of {"x", "y", "color", "size", "opacity"} dicts
    slides=[...],   # required — list of slide dicts
    width=1024,
    height=768,
    padding=0.05,
) -> None           # blocks until window is closed
```

**Slide dict keys**

| Key | Type | Default | Meaning |
|-----|------|---------|--------|
| `title` | `str` | `""` | Window title |
| `zoom` | `"fit"` or `(pan_x, pan_y, zoom)` | `"fit"` | Viewport |
| `layers` | `"all"` or `[1, 2, …]` | `"all"` | 1-based layer indices to show |

**Keyboard shortcuts in the window**
- `1`–`9` — jump to slide
- `→` / `←` — next / previous slide
- drag + scroll — pan / zoom
- `F` — fit visible layers, `Home` — reset origin

> **Note:** `story()` opens a GPU window and blocks until closed — run cells one at a time.

In [1]:
import numpy as np
import justviz as jv

## 1 — Two-cluster reveal

Slide 1 shows everything; slides 2 and 3 isolate each cluster with an auto-fit zoom.

In [2]:
rng = np.random.default_rng(42)

x1 = rng.normal(30, 10, 50_000).astype(np.float32)
y1 = rng.normal(30, 10, 50_000).astype(np.float32)
x2 = rng.normal(70, 10, 50_000).astype(np.float32)
y2 = rng.normal(70, 10, 50_000).astype(np.float32)

jv.story(
    layers=[
        {"x": x1, "y": y1, "color": (0.95, 0.26, 0.21), "size": 2.0, "opacity": 0.6},
        {"x": x2, "y": y2, "color": (0.13, 0.59, 0.95), "size": 2.0, "opacity": 0.6},
    ],
    slides=[
        {"title": "Full dataset",      "zoom": "fit", "layers": "all"},
        {"title": "Cluster A (red)",   "zoom": "fit", "layers": [1]},
        {"title": "Cluster B (blue)",  "zoom": "fit", "layers": [2]},
        {"title": "Both clusters",     "zoom": "fit", "layers": [1, 2]},
    ],
    width=1024, height=768,
)

## 2 — Five-cluster progressive reveal

In [4]:
rng = np.random.default_rng(7)

centres = [(20, 20), (80, 20), (50, 75), (20, 80), (80, 80)]
colors  = [
    (0.95, 0.26, 0.21),
    (0.13, 0.59, 0.95),
    (0.30, 0.69, 0.31),
    (1.00, 0.60, 0.00),
    (0.61, 0.15, 0.69),
]
names = ["Alpha", "Beta", "Gamma", "Delta", "Epsilon"]

layers = []
for (cx, cy), col in zip(centres, colors):
    layers.append({
        "x": rng.normal(cx, 6, 30_000).astype(np.float32),
        "y": rng.normal(cy, 6, 30_000).astype(np.float32),
        "color": col,
        "size": 2.5,
        "opacity": 0.55,
    })

slides = [{"title": "Overview", "zoom": "fit", "layers": "all"}]
for i, name in enumerate(names):
    slides.append({"title": name, "zoom": "fit", "layers": [i + 1]})

jv.story(layers=layers, slides=slides, width=1024, height=768)

## 3 — Explicit viewport (pan + zoom)

Instead of `"fit"`, supply a `(pan_x, pan_y, zoom)` tuple to set an exact camera position.

In [5]:
rng = np.random.default_rng(0)

x_dense = rng.uniform(0, 100, 200_000).astype(np.float32)
y_dense = rng.uniform(0, 100, 200_000).astype(np.float32)

jv.story(
    layers=[{"x": x_dense, "y": y_dense, "color": (0.2, 0.7, 0.9), "size": 1.5}],
    slides=[
        {"title": "Full view",  "zoom": "fit"},
        {"title": "Top-left",   "zoom": (0.0, 0.0, 3.0)},
        {"title": "Centre 5×",  "zoom": (50.0, 50.0, 5.0)},
        {"title": "Bottom-right", "zoom": (100.0, 100.0, 3.0)},
    ],
    width=1024, height=768,
)

## 4 — Annotated data walkthrough

Combine a background layer (all data) with foreground highlight layers.

In [6]:
rng = np.random.default_rng(55)

# Background: all points, faint
x_bg = rng.uniform(0, 100, 500_000).astype(np.float32)
y_bg = rng.uniform(0, 100, 500_000).astype(np.float32)

# Foreground highlights
x_hl1 = rng.normal(25, 5, 10_000).astype(np.float32)
y_hl1 = rng.normal(75, 5, 10_000).astype(np.float32)
x_hl2 = rng.normal(75, 5, 10_000).astype(np.float32)
y_hl2 = rng.normal(25, 5, 10_000).astype(np.float32)

jv.story(
    layers=[
        # Layer 1: background
        {"x": x_bg,  "y": y_bg,  "color": (0.5, 0.5, 0.5), "size": 1.0, "opacity": 0.2},
        # Layer 2: region A highlight
        {"x": x_hl1, "y": y_hl1, "color": (1.0, 0.4, 0.1), "size": 3.5, "opacity": 0.9},
        # Layer 3: region B highlight
        {"x": x_hl2, "y": y_hl2, "color": (0.1, 0.6, 1.0), "size": 3.5, "opacity": 0.9},
    ],
    slides=[
        {"title": "All data (background)",     "zoom": "fit", "layers": [1]},
        {"title": "Region A highlighted",      "zoom": "fit", "layers": [1, 2]},
        {"title": "Region B highlighted",      "zoom": "fit", "layers": [1, 3]},
        {"title": "Both regions highlighted",  "zoom": "fit", "layers": [1, 2, 3]},
        {"title": "Region A zoom",             "zoom": "fit", "layers": [2]},
        {"title": "Region B zoom",             "zoom": "fit", "layers": [3]},
    ],
    width=1200, height=800,
)